In [46]:
import sys
import os

# ensure we're always running from project root
os.chdir(os.path.dirname(os.getcwd()))
sys.path.insert(0, '.')

In [47]:
from src.load import load_data
from src.clean import clean_data
import pandas as pd

df_raw = load_data()
df = clean_data(df_raw)
df = df[df['Customer ID'].notna()][~df['Invoice'].str.startswith('C')]

── Data loaded ──────────────────────────
Rows:              1,033,019
Date range:        2009-12-01 → 2011-12-09
Unique customers:  5,940
Cancellations:     19,100
Null Customer IDs: 235,150
─────────────────────────────────────────
── Cleaning summary ─────────────────────
Rows before: 1,033,019
Rows after:  1,021,374
Removed:     11,645
Outlier rows flagged: 7
─────────────────────────────────────────


C:\Users\TomTurner\AppData\Local\Temp\ipykernel_22816\1847685586.py:7: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df = df[df['Customer ID'].notna()][~df['Invoice'].str.startswith('C')]


## Customer's first purchase month

In [48]:
cohort_df = df.groupby(by='Customer ID', as_index=False)['InvoiceDate'].min()
cohort_df['Cohort'] = cohort_df['InvoiceDate'].dt.to_period('M')
cohort_df.drop(columns=['InvoiceDate'], inplace=True)

cohort_df.head(10)

,Customer ID,Cohort
0,12346,2010-03
1,12347,2010-10
2,12348,2010-09
3,12349,2010-04
4,12350,2011-02
5,12351,2010-11
6,12352,2010-11
7,12353,2010-10
8,12354,2011-04
9,12355,2010-05


## Assign cohort

In [49]:
df = pd.merge(df, cohort_df, on='Customer ID', how='left')
df.sample(10)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalPrice,IsOutlier,Cohort
588503,562608,23159,SET OF 5 PANCAKE DAY MAGNETS,36,2011-08-08 11:45:00,2.08,15502,United Kingdom,74.88,False,2010-06
402584,538592,22113,GREY HEART HOT WATER BOTTLE,2,2010-12-13 11:47:00,3.75,15854,United Kingdom,7.50,False,2009-12
97767,502465,21791,VINTAGE HEADS AND TAILS CARD GAME,9,2010-03-24 15:08:00,1.25,17402,United Kingdom,11.25,False,2009-12
615079,565839,22774,RED DRAWER KNOB ACRYLIC EDWARDIAN,12,2011-09-07 11:46:00,1.25,13186,United Kingdom,15.00,False,2011-08
551108,558043,22742,MAKE YOUR OWN PLAYTIME CARD KIT,6,2011-06-24 13:28:00,2.95,13425,United Kingdom,17.70,False,2010-05
544709,557222,21928,JUMBO BAG SCANDINAVIAN BLUE PAISLEY,10,2011-06-17 13:14:00,2.08,14936,Channel Islands,20.80,False,2010-05
94633,501920,22365,DOOR MAT RESPECTABLE HOUSE,4,2010-03-22 10:23:00,7.49,14040,United Kingdom,29.96,False,2009-12
98486,502594,20829,GLITTER HANGING BUTTERFLY STRING,2,2010-03-25 13:05:00,2.10,16090,United Kingdom,4.20,False,2010-03
139578,507398,48189,DOOR MAT FRIENDSHIP,1,2010-05-09 13:15:00,7.49,17913,United Kingdom,7.49,False,2009-12
544035,557129,21754,HOME BUILDING BLOCK WORD,3,2011-06-16 19:59:00,5.95,17841,United Kingdom,17.85,False,2009-12


## Calculating the period number

In [51]:
df['period'] = (df['InvoiceDate'].dt.to_period('M').astype(int) - df['Cohort'].astype(int))
df

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalPrice,IsOutlier,Cohort,period
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.40,False,2009-12,0
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.00,False,2009-12,0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.00,False,2009-12,0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.80,False,2009-12,0
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.00,False,2009-12,0
...,...,...,...,...,...,...,...,...,...,...,...,...
776594,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680,France,10.20,False,2011-08,4
776595,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680,France,12.60,False,2011-08,4
776596,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680,France,16.60,False,2011-08,4
776597,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680,France,16.60,False,2011-08,4


In [59]:
cohort_data = df.groupby(['Cohort', 'period'])['Customer ID'].nunique().reset_index()
cohort_data.columns = ['Cohort', 'Period', 'Customers']

In [71]:
cohort_data['Period'].min()

np.int64(0)

In [62]:
cohort_matrix = cohort_data.pivot_table(index='Cohort', columns='Period', values='Customers')
cohort_matrix.head()

Period,0,1,2,3,4,5,6,7,8,9,...,15,16,17,18,19,20,21,22,23,24
Cohort,,,,,,,,,,,,,,,,,,,,,
2009-12,951.0,333.0,317.0,404.0,360.0,342.0,358.0,327.0,321.0,344.0,...,288.0,250.0,288.0,270.0,246.0,242.0,298.0,289.0,386.0,187.0
2010-01,368.0,79.0,118.0,116.0,100.0,115.0,99.0,86.0,105.0,120.0,...,56.0,90.0,75.0,71.0,74.0,93.0,73.0,93.0,21.0,NaN
2010-02,375.0,88.0,85.0,110.0,92.0,74.0,72.0,108.0,96.0,104.0,...,75.0,60.0,61.0,54.0,86.0,86.0,62.0,22.0,NaN,NaN
2010-03,441.0,84.0,102.0,107.0,102.0,90.0,109.0,135.0,122.0,48.0,...,75.0,77.0,69.0,78.0,90.0,94.0,34.0,NaN,NaN,NaN
2010-04,294.0,56.0,56.0,47.0,54.0,65.0,81.0,78.0,31.0,32.0,...,46.0,41.0,44.0,53.0,67.0,17.0,NaN,NaN,NaN,NaN


In [66]:
cohort_size = cohort_matrix[0]
retention_matrix = cohort_matrix.divide(cohort_size, axis=0).round(3) * 100
retention_matrix = retention_matrix.iloc[1:]
cohort_matrix = cohort_matrix.iloc[1:]
retention_matrix.head()

Period,0,1,2,3,4,5,6,7,8,9,...,15,16,17,18,19,20,21,22,23,24
Cohort,,,,,,,,,,,,,,,,,,,,,
2010-02,100.0,23.5,22.7,29.3,24.5,19.7,19.2,28.8,25.6,27.7,...,20.0,16.0,16.3,14.4,22.9,22.9,16.5,5.9,NaN,NaN
2010-03,100.0,19.0,23.1,24.3,23.1,20.4,24.7,30.6,27.7,10.9,...,17.0,17.5,15.6,17.7,20.4,21.3,7.7,NaN,NaN,NaN
2010-04,100.0,19.0,19.0,16.0,18.4,22.1,27.6,26.5,10.5,10.9,...,15.6,13.9,15.0,18.0,22.8,5.8,NaN,NaN,NaN,NaN
2010-05,100.0,15.7,16.9,17.6,17.6,25.5,21.2,12.5,5.9,8.2,...,12.5,13.7,16.5,15.3,4.7,NaN,NaN,NaN,NaN,NaN
2010-06,100.0,17.6,18.7,20.6,23.2,28.5,12.7,9.0,8.2,11.2,...,12.4,13.1,20.2,4.9,NaN,NaN,NaN,NaN,NaN,NaN


In [67]:
print("Average retention by period:")
print(f"Period 1:  {retention_matrix[1].mean():.1f}%")
print(f"Period 3:  {retention_matrix[3].mean():.1f}%")
print(f"Period 12: {retention_matrix[12].mean():.1f}%")

Average retention by period:
Period 1:  20.3%
Period 3:  20.2%
Period 12: 16.1%


In [70]:
type(retention_matrix)

pandas.DataFrame